# Exploration des données brutes OpenAgenda

Ce notebook permet d'explorer les données brutes collectées depuis l'API OpenAgenda avant leur preprocessing. L'objectif est de vérifier leur structure et leur qualité, de mesurer les éventuels doublons et d'observer des événements réels pour déterminer ensuite quels champs seront utiles au chatbot et au système RAG. Aucune transformation ni suppression de données n'est réalisée.

## 1. Chargement des données

In [ ]:
import json
from collections import Counter
from pathlib import Path

import pandas as pd

# Retrouver la racine depuis le dossier du notebook ou la racine du projet.
current_directory = Path.cwd().resolve()
project_root = next(
    (directory for directory in (current_directory, *current_directory.parents)
     if (directory / 'App' / 'Data' / 'raw').is_dir()),
    None,
)
if project_root is None:
    raise FileNotFoundError('Dossier App/Data/raw introuvable : lancez le notebook depuis ce projet.')

raw_directory = project_root / 'App' / 'Data' / 'raw'
json_files = list(raw_directory.glob('openagenda_*.json'))
if not json_files:
    raise FileNotFoundError(f'Aucun fichier openagenda_*.json dans {raw_directory}.')

# La date de modification détermine le fichier le plus récent.
input_path = max(json_files, key=lambda path: (path.stat().st_mtime_ns, path.name))
with input_path.open(encoding='utf-8') as input_file:
    payload = json.load(input_file)

if not isinstance(payload, dict):
    raise ValueError('Le JSON doit contenir un objet avec une liste events.')
events = payload.get('events')
if not isinstance(events, list) or any(not isinstance(event, dict) for event in events):
    raise ValueError('Le champ events doit être une liste d’objets JSON.')

print(f'Fichier utilisé : {input_path.name}')
print(f'Événements chargés : {len(events):,}')

## 2. Vérification des doublons

Les doublons sont les occurrences supplémentaires d'un même UID. Les UID absents ou invalides sont comptés séparément et exclus de ce calcul. Aucun événement n'est supprimé.

In [ ]:
uids = []
missing_uid_count = 0
invalid_uid_count = 0
for event in events:
    uid = event.get('uid')
    if uid is None or (isinstance(uid, str) and not uid.strip()):
        missing_uid_count += 1
    elif type(uid) not in (str, int):
        invalid_uid_count += 1
    else:
        # Comparer 123 et '123' comme le même identifiant, sans modifier les événements.
        uids.append(str(uid))

uid_counts = Counter(uids)
duplicate_count = len(uids) - len(uid_counts)
print(f'Événements bruts : {len(events):,}')
print(f'Événements sans UID : {missing_uid_count:,}')
print(f'Événements avec un type d’UID invalide : {invalid_uid_count:,}')
print(f'UID uniques parmi les événements avec UID exploitable : {len(uid_counts):,}')
print(f'Doublons (occurrences supplémentaires) : {duplicate_count:,}')

## 3. Exploration des champs

Cette liste rassemble les clés de premier niveau rencontrées dans tous les événements. Les structures imbriquées seront visibles dans les exemples suivants.

In [ ]:
fields = sorted({field for event in events for field in event})
print(f'Nombre de champs rencontrés : {len(fields)}')
for field in fields:
    print(f'- {field}')

## 4. Trois événements au hasard

Le tirage utilise `random_state=42` pour retrouver les mêmes exemples tant que le fichier et l'ordre des événements restent identiques. S'il y a moins de trois événements, tous sont affichés.

In [ ]:
# Tirer seulement les positions avec Pandas ; les objets JSON restent intacts.
sample_positions = pd.Series(range(len(events)), dtype='int64').sample(
    n=min(3, len(events)), random_state=42
)
if not events:
    print('Aucun événement disponible pour les exemples.')
for example_number, position in enumerate(sample_positions, start=1):
    print(f'\n---------- Exemple {example_number} — position {position} ----------')
    print(json.dumps(events[position], ensure_ascii=False, indent=2))